---
## 1. Setup & Load


In [ ]:
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib seaborn \
    scikit-surprise optuna wandb requests
print('Done')


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, ast, json, time, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import requests

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#F8F7F4',
    'axes.grid':True,'grid.color':'#E0DED8',
    'axes.spines.top':False,'axes.spines.right':False,
})
C_BEFORE='#A32D2D'; C_AFTER='#0F6E56'; C_FLAG='#BA7517'; C_PURPLE='#534AB7'

os.makedirs('data',exist_ok=True)
os.makedirs('plots',exist_ok=True)
os.makedirs('models',exist_ok=True)
np.random.seed(42); random.seed(42)

def validate(condition, label, expected, actual):
    status = '✅ PASS' if condition else '❌ FAIL'
    print(f'{status}  {label}')
    print(f'         Expected: {expected}')
    print(f'         Actual:   {actual}')
    if not condition:
        raise AssertionError(f'Validation failed: {label}')

print('Setup complete')


In [ ]:
# ── Load raw recipes ──────────────────────────────────────────────────────────
df_recipes_raw = pd.read_csv('data/RAW_recipes.csv')
print(f'Loaded RAW_recipes.csv: {len(df_recipes_raw):,} rows')

# Parse nutrition column
# Order: [calories, total_fat_g, sugar_g, sodium_mg, protein_g, sat_fat_g, carbs_g]
NUTRITION_COLS = ['calories','total_fat_g','sugar_g','sodium_mg',
                   'protein_g','sat_fat_g','carbs_g']

def parse_nutrition(s):
    try:
        v = ast.literal_eval(str(s))
        return v if len(v)==7 else [np.nan]*7
    except: return [np.nan]*7

nutr = pd.DataFrame(
    df_recipes_raw['nutrition'].apply(parse_nutrition).tolist(),
    columns=NUTRITION_COLS)
df_recipes_raw = pd.concat(
    [df_recipes_raw.drop(columns=['nutrition']), nutr], axis=1)

df_recipes_raw['tags_list'] = df_recipes_raw['tags'].apply(
    lambda s: ast.literal_eval(str(s)) if pd.notna(s) else [])

for col in NUTRITION_COLS + ['minutes','n_steps','n_ingredients']:
    if col in df_recipes_raw.columns:
        df_recipes_raw[col] = pd.to_numeric(df_recipes_raw[col], errors='coerce')

# ── Validation ────────────────────────────────────────────────────────────────
validate(len(df_recipes_raw) > 10000,
         'Recipe count reasonable',
         '> 10,000 recipes',
         f'{len(df_recipes_raw):,}')
validate(all(c in df_recipes_raw.columns for c in NUTRITION_COLS),
         'All nutrition columns parsed',
         'All 7 columns present',
         f'{sum(c in df_recipes_raw.columns for c in NUTRITION_COLS)}/7')
validate(df_recipes_raw['calories'].notna().mean() > 0.5,
         'Calories column mostly populated',
         '> 50% non-null',
         f'{df_recipes_raw["calories"].notna().mean():.1%}')

print(f'\nRecipe columns: {list(df_recipes_raw.columns)}')
print(df_recipes_raw[['name'] + NUTRITION_COLS].head(3).to_string(index=False))


In [ ]:
# ── Load pre-split interaction files ─────────────────────────────────────────
df_train_raw = pd.read_csv('data/interactions_train.csv')
df_test_raw  = pd.read_csv('data/interactions_test.csv')
df_valid_raw = pd.read_csv('data/interactions_validation.csv')

print(f'Train raw: {len(df_train_raw):,}  cols: {list(df_train_raw.columns)}')
print(f'Test  raw: {len(df_test_raw):,}')
print(f'Valid raw: {len(df_valid_raw):,}')
print(df_train_raw.head(3).to_string(index=False))


In [ ]:
# ── Standardise interactions ──────────────────────────────────────────────────
# The correct ID columns are 'user_id' and 'recipe_id' (original Food.com IDs)
# 'u' and 'i' are remapped sequential indices — do NOT use them
# Confirmed: recipe_id has 100% overlap with RAW_recipes['id']

def prepare_interactions(df):
    df = df.copy()
    df['user_id']   = df['user_id'].astype(str)
    df['recipe_id'] = df['recipe_id'].astype(str)
    df['rating']    = 1   # binary implicit feedback
    cols = ['user_id','recipe_id','rating']
    if 'date' in df.columns: cols.append('date')
    return df[cols]

# Combine train + validation for training (standard practice)
df_train_inter = prepare_interactions(
    pd.concat([df_train_raw, df_valid_raw], ignore_index=True))
df_test_inter  = prepare_interactions(df_test_raw)

# Remove duplicate (user, recipe) pairs
df_train_inter = df_train_inter.drop_duplicates(
    subset=['user_id','recipe_id']).reset_index(drop=True)
df_test_inter  = df_test_inter.drop_duplicates(
    subset=['user_id','recipe_id']).reset_index(drop=True)

# Check ID overlap with RAW_recipes
raw_ids       = set(df_recipes_raw['id'].astype(str))
train_rids    = set(df_train_inter['recipe_id'])
test_rids     = set(df_test_inter['recipe_id'])
train_overlap = len(train_rids & raw_ids) / len(train_rids)
test_overlap  = len(test_rids  & raw_ids) / len(test_rids)

print(f'Train interactions: {len(df_train_inter):,}')
print(f'Test  interactions: {len(df_test_inter):,}')
print(f'Unique users (train): {df_train_inter["user_id"].nunique():,}')
print(f'Unique recipes (train): {df_train_inter["recipe_id"].nunique():,}')
print(f'Train recipe overlap with RAW_recipes: {train_overlap:.1%}')
print(f'Test  recipe overlap with RAW_recipes: {test_overlap:.1%}')

# ── Validation ────────────────────────────────────────────────────────────────
validate(len(df_train_inter) > 50000,
         'Train interactions sufficient',
         '> 50,000',
         f'{len(df_train_inter):,}')
validate(len(df_test_inter) > 5000,
         'Test interactions sufficient',
         '> 5,000',
         f'{len(df_test_inter):,}')
validate(df_train_inter['user_id'].nunique() > 1000,
         'Sufficient unique users in train',
         '> 1,000',
         f'{df_train_inter["user_id"].nunique():,}')
validate(train_overlap > 0.8,
         'Train recipe IDs match RAW_recipes',
         '> 80% overlap',
         f'{train_overlap:.1%}')
validate(test_overlap > 0.8,
         'Test recipe IDs match RAW_recipes',
         '> 80% overlap',
         f'{test_overlap:.1%}')
